# Step 10: Basic Workflows

        _Learner Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Intermediate**

        ## Learning Objectives
        - [ ] Create a sequential workflow with typed executors.
- [ ] Connect nodes using `WorkflowBuilder`.
- [ ] Run the workflow on sample input.
- [ ] Read workflow outputs as a structured pipeline result.

        ## Prerequisites
        - Step 9 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Workflows make dataflow explicit. Instead of one agent doing everything, we connect small executors into a graph that can be reasoned about step by step.

Expected output and validation notes:

Expected output snapshot:

- The workflow returns a list containing one final report string


## Part 2: Core Implementation

### Build a three-step workflow


In [ ]:
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler

class ExtractExecutor(Executor):
    @handler
    async def process(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"KEY IDEAS: {text[:120]}")

class AnalyzeExecutor(Executor):
    @handler
    async def process(self, extracted: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"ANALYSIS: {extracted}")

class ReportExecutor(Executor):
    @handler
    async def process(self, analysis: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"REPORT: {analysis}")

extract = ExtractExecutor(id="extract")
analyze = AnalyzeExecutor(id="analyze")
report = ReportExecutor(id="report")

workflow = WorkflowBuilder(start_executor=extract).add_chain([extract, analyze, report]).build()


## Part 2: Core Implementation

### Run the workflow


In [ ]:
run_result = await workflow.run(
    "The engineering team wants a course that moves from single agents to orchestration."
)
print(run_result.get_outputs())


## Part 3: Hands-On Exercises

Add a fourth executor that reformats the report before it becomes the final output.

Try the exercise yourself before looking at the solutions in Part 4.


In [ ]:
# TODO: add one more executor to the chain
# TODO: rebuild the workflow and run it again


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
class DraftReportExecutor(Executor):
    @handler
    async def process(self, analysis: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"DRAFT REPORT: {analysis}")

class FormatExecutor(Executor):
    @handler
    async def process(self, report_text: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(report_text.replace("DRAFT REPORT:", "FINAL REPORT:"))

draft_report = DraftReportExecutor(id="draft_report")
formatter = FormatExecutor(id="formatter")
workflow_v2 = WorkflowBuilder(start_executor=extract).add_chain([extract, analyze, draft_report, formatter]).build()
print((await workflow_v2.run("Course design should stay progressive and hands-on.")).get_outputs())


## Part 5: Best Practices & Tips

        - Keep executor responsibilities narrow and typed.
- Prefer chains for simple linear flows before using branching APIs.
- Inspect outputs at each step when debugging graph behavior.


## Summary & Key Takeaways

You created a sequential typed workflow and ran it end to end.


## Additional Resources

        - `agent_framework WorkflowBuilder docs`
- `Step 11 notebook`
